# Plan 5 Phase B — Colab run 2 (L4 or A100)

Plan 5 lives in the `plan_5_spatial_equity/` folder of the repository. The first cell clones `main` (or, if the repository is already cloned, pulls `main`) and changes into that folder. Every later cell and every path below is relative to it. Run 2 writes new files and leaves run 1 in place.

Features are `ml_features` from `feature_manifest.json`. BGE-M3 uses right padding and the first non-pad token as CLS. LoRA stays 4-bit, `r=8`, 40 steps, MRL width 64, and is fit on every training-block row. Each text arm trains GATv2 and TransformerConv for seeds 42–51, inductive and transductive.

Runtime: **L4** or **A100**.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Lc-wing/data_science_project.git"
BRANCH = "main"
PLAN_DIR = "plan_5_spatial_equity"
ENTRY = "scripts/colab_phase_b.py"


def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)


top = git("rev-parse", "--show-toplevel").stdout.strip()
if top and ((Path(top) / PLAN_DIR / ENTRY).exists() or (Path(top) / ENTRY).exists()):
    top = Path(top)
    for args in (("fetch", "origin", BRANCH), ("checkout", BRANCH), ("pull", "origin", BRANCH)):
        done = git(*args, cwd=top)
        assert done.returncode == 0, done.stderr
else:
    top = Path("data_science_project").resolve()
    if not (top / ".git").exists():
        done = git("clone", "--branch", BRANCH, REPO, str(top))
        assert done.returncode == 0, done.stderr

# A checkout from before the folder split keeps Plan 5 at the repository root.
plan_root = top / PLAN_DIR if (top / PLAN_DIR / ENTRY).exists() else top
assert (plan_root / ENTRY).exists(), f"{ENTRY} not found under {plan_root}"
os.chdir(plan_root)
print(Path.cwd())


In [ ]:
import torch

assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU (L4 or A100)."
print(torch.cuda.get_device_name(0))
!pip install -q h3 torch-geometric transformers peft bitsandbytes accelerate


In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
!python scripts/colab_phase_b.py --allow-weight-download


In [ ]:
from pathlib import Path

print(Path("outputs/tables/phase_b_comparison_run2.md").read_text())
